In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted!")

Mounted at /content/drive
Google Drive mounted!


In [2]:
import pandas as pd
import numpy as np
import os

INPUT_DIR  = "/content/drive/MyDrive/insider_threat/data"
OUTPUT_DIR = "/content/drive/MyDrive/insider_threat/data"

print("Libraries loaded!")
print(f"Reading from: {INPUT_DIR}")

Libraries loaded!
Reading from: /content/drive/MyDrive/insider_threat/data


In [3]:
print("Loading raw CSV files...")

logon_df  = pd.read_csv(f"{INPUT_DIR}/logon.csv")
file_df   = pd.read_csv(f"{INPUT_DIR}/file.csv")
email_df  = pd.read_csv(f"{INPUT_DIR}/email.csv")
device_df = pd.read_csv(f"{INPUT_DIR}/device.csv")

print(f"Logon rows  : {len(logon_df):,}")
print(f"File rows   : {len(file_df):,}")
print(f"Email rows  : {len(email_df):,}")
print(f"Device rows : {len(device_df):,}")
print("Files loaded!")

Loading raw CSV files...
Logon rows  : 65,400
File rows   : 196,304
Email rows  : 278,521
Device rows : 1,760
Files loaded!


In [4]:
print("Tagging and combining data sources...")

logon_df["source"]  = "logon"
file_df["source"]   = "file"
email_df["source"]  = "email"
device_df["source"] = "device"

def slim(df, extra_cols=None):
    base = ["id", "date", "user", "activity", "source"]
    cols = base + (extra_cols or [])
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()

logon_slim  = slim(logon_df,  extra_cols=["pc"])
file_slim   = slim(file_df,   extra_cols=["filename"])
email_slim  = slim(email_df,  extra_cols=["to", "size", "attachments"])
device_slim = slim(device_df, extra_cols=["pc"])

master = pd.concat([logon_slim, file_slim, email_slim, device_slim],
                   ignore_index=True)
print(f"Combined master log: {len(master):,} rows")
print("Done!")

Tagging and combining data sources...
Combined master log: 541,985 rows
Done!


In [5]:
print("Parsing datetime column...")

master["datetime"] = pd.to_datetime(master["date"],
                                    format="%m/%d/%Y %H:%M:%S",
                                    errors="coerce")

bad_dates = master["datetime"].isna().sum()
if bad_dates > 0:
    print(f"WARNING: {bad_dates} rows had unparseable dates - dropping them")
    master = master.dropna(subset=["datetime"])

print(f"Date range: {master['datetime'].min().date()} to {master['datetime'].max().date()}")
print("Done!")

Parsing datetime column...
Date range: 2023-01-01 to 2023-03-31
Done!


In [6]:
print("Adding time-based helper columns...")

master["hour"]           = master["datetime"].dt.hour
master["day_of_week"]    = master["datetime"].dt.dayofweek
master["date_only"]      = master["datetime"].dt.date
master["is_weekend"]     = master["day_of_week"] >= 5
master["is_after_hours"] = (master["hour"] < 7) | (master["hour"] >= 19)
master["is_off_hours"]   = master["is_weekend"] | master["is_after_hours"]

before = len(master)
master = master.drop_duplicates(subset=["id", "user", "datetime"])
after  = len(master)
if before != after:
    print(f"Removed {before - after:,} duplicate rows")

print("Added: hour, day_of_week, is_weekend, is_after_hours, is_off_hours")
print("Done!")

Adding time-based helper columns...
Added: hour, day_of_week, is_weekend, is_after_hours, is_off_hours
Done!


In [7]:
print("Saving master log to Google Drive...")

master = master.sort_values(["user", "datetime"]).reset_index(drop=True)
master.to_csv(f"{OUTPUT_DIR}/master_log.csv", index=False)

print("=" * 50)
print("  PREPROCESSING COMPLETE")
print("=" * 50)
print(f"  Total rows      : {len(master):,}")
print(f"  Unique users    : {master['user'].nunique():,}")
print(f"  Activity types  : {master['activity'].nunique()}")
print(f"  Off-hours events: {master['is_off_hours'].sum():,} ({master['is_off_hours'].mean()*100:.1f}%)")
print("=" * 50)
print("master_log.csv saved to Google Drive!")
print("Next step: Run 03_feature_engineering notebook!")
master.head()

Saving master log to Google Drive...
  PREPROCESSING COMPLETE
  Total rows      : 541,985
  Unique users    : 500
  Activity types  : 7
  Off-hours events: 145,941 (26.9%)
master_log.csv saved to Google Drive!
Next step: Run 03_feature_engineering notebook!


,id,date,user,activity,source,pc,filename,to,size,attachments,datetime,hour,day_of_week,date_only,is_weekend,is_after_hours,is_off_hours
0,E0000007,01/01/2023 07:09:15,U0001,Send,email,NaN,NaN,user_91@company.com,49284.0,0.0,2023-01-01 07:09:15,7,6,2023-01-01,True,False,True
1,F0000002,01/01/2023 10:44:42,U0001,write,file,NaN,file_9188.xlsx,NaN,NaN,NaN,2023-01-01 10:44:42,10,6,2023-01-01,True,False,True
2,F0000000,01/01/2023 11:48:58,U0001,open,file,NaN,file_5368.xlsx,NaN,NaN,NaN,2023-01-01 11:48:58,11,6,2023-01-01,True,False,True
3,E0000003,01/01/2023 12:21:49,U0001,Send,email,NaN,NaN,user_355@company.com,332223.0,1.0,2023-01-01 12:21:49,12,6,2023-01-01,True,False,True
4,F0000001,01/01/2023 13:36:13,U0001,copy,file,NaN,file_9239.txt,NaN,NaN,NaN,2023-01-01 13:36:13,13,6,2023-01-01,True,False,True
